# HeyDesk's Bangla voice on a free GPU

A GPU stand-in for `tts_server/server.py`, for demos only. It serves the same
`POST /speak {text}` → 16-bit PCM (`X-Sample-Rate` header), so HeyDesk changes nothing but a URL.

**Run it on Kaggle:**

1. kaggle.com → **Create → New notebook → File → Import notebook**, and pick this file.
2. Right panel, **Settings**: *Accelerator* **GPU T4 x2**, *Internet* **on**
   (both need a phone-verified Kaggle account).
3. Pick `MODEL` and `VOICE` in the first cell, then **Run all**. The first run downloads
   the model (BanglaBox about 3.5 GB) and takes about 5 minutes.
4. The last cell plays a test sentence and prints a line like
   `BANGLA_TTS_URL=https://….trycloudflare.com`. Put it in HeyDesk's `.env` and restart HeyDesk.
5. Keep this tab open during the demo. Afterwards, **Run → Stop session** (Kaggle gives 30 GPU hours a week).

To switch models: change `MODEL`, **Run → Restart & clear cell outputs**, then **Run all** again.
The two need different `transformers` versions, so one kernel runs one model. The URL changes on
every run.

The URL is public while the session runs: anyone who has it can make speech. Colab works too
technically, but Colab's terms forbid serving from a notebook. Use Kaggle.


In [ ]:
MODEL = "banglabox"  # or "omnivoice"
VOICE = "male"  # or "female": a reference clip from BanglaBox, used by both models
PORT = 8010

In [ ]:
import subprocess
import sys

# Kaggle's own torch stays; these are what each model needs on top of it.
PACKAGES = {
    "banglabox": [
        "transformers==4.46.3",
        "tokenizers==0.20.3",
        "peft==0.17.1",
        "diffusers==0.29.0",
        "conformer==0.3.2",
        "s3tokenizer==0.3.0",
        "omegaconf==2.3.1",
        "einops==0.8.2",
        "librosa==0.11.0",
        "safetensors==0.5.3",
        "bnnumerizer",
        "bnunicodenormalizer",
        "resemble-perth==1.0.1",  # Chatterbox imports these two; BanglaBox's list leaves them out
        "setuptools<81",  # perth needs pkg_resources
    ],
    "omnivoice": ["omnivoice"],
}
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PACKAGES[MODEL], "fastapi", "uvicorn"],
    check=True,
)

In [ ]:
import os

import numpy as np
import torch
from huggingface_hub import snapshot_download

assert torch.cuda.is_available() or os.environ.get("ALLOW_CPU"), "Settings → Accelerator: GPU T4 x2"

# BanglaBox's repo has the reference voices both models clone, and BanglaBox itself.
repo = snapshot_download(
    "Banglabox/chatterbox-bangla-tts",
    allow_patterns=["voices/*"]
    if MODEL == "omnivoice"
    else ["voices/*", "inference/**", "checkpoint/**", "pretrained_models/*"],
)
REFERENCE = os.path.join(repo, "voices", f"ref_{VOICE}.wav")

if MODEL == "banglabox":
    import sys

    # infer.py wants pretrained_models/, adapter/ and NEW_VOCAB_SIZE.txt side by side.
    kit = os.path.abspath("banglabox")
    os.makedirs(kit, exist_ok=True)
    for name, source in {
        "pretrained_models": "pretrained_models",
        "adapter": "checkpoint/adapter",
        "NEW_VOCAB_SIZE.txt": "inference/NEW_VOCAB_SIZE.txt",
    }.items():
        if not os.path.lexists(os.path.join(kit, name)):
            os.symlink(os.path.join(repo, source), os.path.join(kit, name))
    sys.path.insert(0, os.path.join(repo, "inference"))
    from infer import BanglaTTS

    banglabox = BanglaTTS(kit)

    def speak_wav(text: str) -> tuple[np.ndarray, int]:
        return banglabox.tts(text, REFERENCE)

else:
    from omnivoice import OmniVoice

    cuda = torch.cuda.is_available()
    omnivoice = OmniVoice.from_pretrained(
        "k2-fsa/OmniVoice",
        device_map="cuda:0" if cuda else "cpu",
        dtype=torch.float16 if cuda else torch.float32,
    )
    omnivoice.load_asr_model()  # Whisper writes the reference clip's transcript, once
    prompt = omnivoice.create_voice_clone_prompt(ref_audio=REFERENCE)
    print("Whisper heard the reference as:", prompt.ref_text)

    def speak_wav(text: str) -> tuple[np.ndarray, int]:
        return omnivoice.generate(text=text, language="bn", voice_clone_prompt=prompt)[0], 24000


speak_wav("পরীক্ষা।")  # warm up: the first sentence is always slow
print("model ready:", MODEL, VOICE)

In [ ]:
import asyncio
import threading
import time

import uvicorn
from fastapi import FastAPI, Response
from pydantic import BaseModel, Field

app = FastAPI()
lock = asyncio.Lock()  # one sentence at a time, like tts_server/server.py


class Speak(BaseModel):
    text: str = Field(min_length=1, max_length=1000)


@app.post("/speak")
async def speak(body: Speak) -> Response:
    """The sentence as raw 16-bit mono PCM; the rate is in `X-Sample-Rate`."""
    async with lock:
        start = time.perf_counter()
        wav, rate = await asyncio.to_thread(speak_wav, body.text)
    took = time.perf_counter() - start
    print(f"{len(wav) / rate:.1f} s of speech in {took:.1f} s")
    pcm = (np.clip(np.asarray(wav), -1, 1) * 32767).astype("<i2").tobytes()
    return Response(pcm, media_type="audio/L16", headers={"X-Sample-Rate": str(rate)})


threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "127.0.0.1", "port": PORT, "log_level": "warning"},
    daemon=True,
).start()

In [ ]:
import re
import subprocess
import urllib.request
from pathlib import Path

# A Cloudflare quick tunnel: a public https URL for this notebook's port, no account needed.
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared",
    )
    os.chmod("cloudflared", 0o755)
log = open("cloudflared.log", "w")  # noqa: SIM115 (the tunnel writes to it while it runs)
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=log,
    stderr=log,
)
URL = None
for _ in range(60):
    found = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com", Path("cloudflared.log").read_text()
    )
    if found:
        URL = found.group(0)
        break
    time.sleep(1)
assert URL, "no tunnel URL yet: see cloudflared.log"

In [ ]:
import json

from IPython.display import Audio, display

# One sentence, sent the way HeyDesk will send it. To the server here, not the tunnel:
# some notebooks can't reach a trycloudflare address from inside ("Network is unreachable").
request = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/speak",
    data=json.dumps({"text": "আপনার অর্ডারের মোট দাম তিনশো ষাট টাকা। ঠিক আছে?"}).encode(),
    headers={"Content-Type": "application/json"},
)
start = time.perf_counter()
with urllib.request.urlopen(request, timeout=60) as response:
    rate = int(response.headers["X-Sample-Rate"])
    pcm = np.frombuffer(response.read(), dtype="<i2")
print(f"on this machine: {len(pcm) / rate:.1f} s of speech in {time.perf_counter() - start:.1f} s")
display(Audio(pcm.astype(np.float32) / 32767, rate=rate))
print("\nPut this line in HeyDesk's .env, then restart HeyDesk:\n")
print(f"BANGLA_TTS_URL={URL}")